# 03 · Normalization with uMAIA

Following the tutorial notebook [`03_normalization_umaia_student.ipynb`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/blob/main/notebooks/level1/03_normalization_umaia_student.ipynb).

**Kernel:** `cajal-umaia`

**What I'm doing here**
- Batch effects and the bimodal log histogram (anchor low, drift high)
- The uMAIA model: rank-1 batch shift gamma_a * lambda_c, fit by MAP via SVI
- Run initialize -> normalize -> transform, unroll the F^-1(G(x)) histogram matching
- Per-lipid 0-1 scaling, then save `03_normalized.h5ad` with layers['umaia']

**My notes / what surprised me**
- ...

In [ ]:
# ---- setup for notebook 03: runs on the cajal-umaia kernel ---------------------------------

# uMAIA's model is written in JAX. JAX will go hunting for a GPU (and CUDA) if I let it, so I
# pin it to the CPU. This MUST happen before anything imports jax, because JAX reads this
# setting once, at import time. That's why it's the very first thing in the notebook.
import os
os.environ["JAX_PLATFORMS"] = "cpu"

# Same path helper as every other notebook
from lipid_project import paths
paths.ensure_dirs()   # make sure data/derived/ exists, since this notebook writes 03_normalized.h5ad
paths.summary()       # check that 02_annotated is listed under "stages built"

import numpy as np
import matplotlib.pyplot as plt
import anndata as ad

import jax                       # the numerical engine uMAIA's model runs on
import uMAIA                     # the normalization package (cloned into external/uMAIA by setup.sh)
import cajal_lipidomics as cl    # tutorial helpers (installed with --no-deps in this env)

# Should print "cpu". If I get ModuleNotFoundError: No module named 'uMAIA', I'm on the wrong
# kernel: this notebook needs "cajal-umaia", not "cajal-lipidomics".
print("ready. JAX on", jax.devices()[0].platform)

## load

In [ ]:
# ---- load the previous stage --------------------------------------------------------------
# My NB02 output: both sections stacked in one AnnData, still on the RAW intensity scale,
# with lipid names in var. The whole point of this notebook is to make the same lipid
# comparable across the two sections (control vs pregnant were acquired separately).
adata = ad.read_h5ad(paths.stage("annotated"))

print(f"{adata.n_obs:,} pixels x {adata.n_vars} lipids")
print("conditions:", dict(adata.obs["Condition"].value_counts()))

# Reminder to self: with only 1 control vs 1 pregnant section, "batch effect" and "biology"
# are tangled together. uMAIA can't fully separate them. Keep that in mind when reading results.

# ---- output -----------------------------------------------------------------------------------
# I keep the raw X untouched and add the normalized values as a separate LAYER, on the native
# (un-logged) scale. Every later notebook reads layers["umaia"]:
#   adata.layers["umaia"] = umaia_native
#   adata.write_h5ad(paths.stage("normalized"))   ->  data/derived/03_normalized.h5ad

## my work

In [ ]:
# my code from here on, one small step per cell, and look at the output each time